# 06 · Kiểm đầu vào và tokenizer thực tế
Đối chiếu Word 2.3.2, 3.3 và PPT slide 6/14. Chỉ tải tokenizer nhỏ; không tải hoặc huấn luyện mô hình.

In [1]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/notebook_support.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
EXPERIMENT = ROOT/'experiments/best_epochs_20261103'
MAIN = EXPERIMENT/'main_results'
ERRORS = ROOT/'experiments/threshold_error_20261110'
pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 40)
pd.set_option('display.float_format', lambda x: f'{x:.6f}')
print('Báo cáo Final v7.1 · chỉ đánh giá từ dữ liệu/điểm đã lưu, không huấn luyện lại.')
from src.notebook_support import artifacts_root
from src.datasets.goemotions import load_goemotions,download,sha256,multi_hot
from tokenizers import Tokenizer
frames, labels, _ = load_goemotions(artifacts_root(),write_metadata=False)
tokenizers={};tokenizer_rows=[]
for architecture in ['bert','roberta','distilbert']:
    meta=json.loads((MAIN/f'training_receipts/{architecture}/seed_42/run_metadata.json').read_text(encoding='utf-8'))
    model_id,revision=meta['checkpoint'],meta['checkpoint_revision']
    path=download(f'https://huggingface.co/{model_id}/resolve/{revision}/tokenizer.json',ROOT/f'data/tokenizers/{architecture}/{revision}/tokenizer.json')
    tokenizer=Tokenizer.from_file(str(path));tokenizer.no_truncation();tokenizer.no_padding()
    tokenizers[architecture]=tokenizer
    tokenizer_rows.append({'mô hình':architecture,'checkpoint':model_id,'revision':revision,'sha256':sha256(path)})
display(pd.DataFrame(tokenizer_rows))

Báo cáo Final v7.1 · chỉ đánh giá từ dữ liệu/điểm đã lưu, không huấn luyện lại.


,mô hình,checkpoint,revision,sha256
0,bert,google-bert/bert-base-cased,cd5ef92a9fb2f889e972770a36d4ed042daf221e,5b3360be30cdf39081153cf9d30bfb0503c011462fc9f7...
1,roberta,FacebookAI/roberta-base,e2da8e2f811d1448a5b465c236feacd80ffbac7b,847bbeab6174d66a88898f729d52fa8d355fafe1bea101...
2,distilbert,distilbert/distilbert-base-uncased,12040accade4e8a0f71eabdb258fecc2e7e948be,ce64fce797c24f68df90b40a3f74f579b336a493db14bd...


## 1. Một câu được biến thành những token và ID nào?
ID chỉ là vị trí trong từ vựng riêng, không mang thứ tự lớn nhỏ về cảm xúc. Chữ hoa được giữ hoặc chuẩn hóa theo từng tokenizer.

In [2]:
sentence='Thank you, I am happy!'
rows=[]
for architecture,tok in tokenizers.items():
    encoded=tok.encode(sentence)
    rows.append({'mô hình':architecture,'token':encoded.tokens,'ID':encoded.ids,'mask':encoded.attention_mask,'số token':len(encoded.ids)})
display(pd.DataFrame(rows))
print('BERT cased: token happy có ID',tokenizers['bert'].token_to_id('happy'))

,mô hình,token,ID,mask,số token
0,bert,"[[CLS], Thank, you, ,, I, am, happy, !, [SEP]]","[101, 4514, 1128, 117, 146, 1821, 2816, 106, 102]","[1, 1, 1, 1, 1, 1, 1, 1, 1]",9
1,roberta,"[<s>, Thank, Ġyou, ,, ĠI, Ġam, Ġhappy, !, </s>]","[0, 13987, 47, 6, 38, 524, 1372, 328, 2]","[1, 1, 1, 1, 1, 1, 1, 1, 1]",9
2,distilbert,"[[CLS], thank, you, ,, i, am, happy, !, [SEP]]","[101, 4067, 2017, 1010, 1045, 2572, 3407, 999,...","[1, 1, 1, 1, 1, 1, 1, 1, 1]",9


BERT cased: token happy có ID 2816


## 2. Đo P95, phần bị cắt và [UNK] trên đúng dữ liệu
Đếm câu có ít nhất một [UNK], không phải số dòng bị mất. Max_length=128 tính cả token đặc biệt. Bảng test chỉ mô tả, không dùng chọn độ dài.

In [3]:
rows=[]
for architecture,tok in tokenizers.items():
    unk=tok.token_to_id('[UNK]')
    for split,frame in frames.items():
        encoded=tok.encode_batch(frame.text.tolist())
        lengths=np.array([len(e.ids) for e in encoded])
        count_unknown=sum(unk is not None and unk in e.ids for e in encoded)
        rows.append({'mô hình':architecture,'split':split,'số câu':len(frame),
                     'P95 token':np.quantile(lengths,.95),'câu >128 token':int((lengths>128).sum()),
                     'câu có [UNK]':count_unknown})
input_audit=pd.DataFrame(rows);display(input_audit)
assert input_audit[input_audit.split=='train']['P95 token'].tolist()==[35,33,34]
expected={'bert':[869,117,112],'distilbert':[847,115,108],'roberta':[0,0,0]}
for architecture,counts in expected.items():
    assert input_audit[input_audit['mô hình']==architecture]['câu có [UNK]'].tolist()==counts
print('PASS: P95 và số câu [UNK] khớp Word, không bỏ dòng dữ liệu.')

,mô hình,split,số câu,P95 token,câu >128 token,câu có [UNK]
0,bert,train,43410,35.000000,1,869
1,bert,validation,5426,36.000000,0,117
2,bert,test,5427,35.000000,0,112
3,roberta,train,43410,33.000000,4,0
4,roberta,validation,5426,33.000000,0,0
5,roberta,test,5427,33.000000,1,0
6,distilbert,train,43410,34.000000,1,847
7,distilbert,validation,5426,34.000000,0,115
8,distilbert,test,5427,34.000000,0,108


PASS: P95 và số câu [UNK] khớp Word, không bỏ dòng dữ liệu.


## 3. Padding, mask và ba trục B × L × 768
Hai câu được đệm tới độ dài L của batch; mask=0 ở vị trí đệm. Encoder nhận B×L ID, cho B×L×768 hidden; head phân loại cả câu cho B×28 logit.

In [4]:
tok=tokenizers['bert']
tok.enable_truncation(max_length=128)
tok.enable_padding(pad_id=tok.token_to_id('[PAD]'),pad_token='[PAD]')
batch=tok.encode_batch(['Thank you.','Thank you, I am happy!'])
ids=np.array([e.ids for e in batch]);mask=np.array([e.attention_mask for e in batch])
display(pd.DataFrame({'câu':['Thank you.','Thank you, I am happy!'], 'input_ids':ids.tolist(),'attention_mask':mask.tolist()}))
B,L=ids.shape
display(pd.Series({'input_ids':ids.shape,'attention_mask':mask.shape,'hidden (hình dạng lý thuyết)':(B,L,768),'logits (hình dạng lý thuyết)':(B,28)}))
print('Không chạy encoder trong cell này; hidden/logit ở đây là hình dạng, không phải đầu ra đo được.')

,câu,input_ids,attention_mask
0,Thank you.,"[101, 4514, 1128, 119, 102, 0, 0, 0, 0]","[1, 1, 1, 1, 1, 0, 0, 0, 0]"
1,"Thank you, I am happy!","[101, 4514, 1128, 117, 146, 1821, 2816, 106, 102]","[1, 1, 1, 1, 1, 1, 1, 1, 1]"


input_ids                            (2, 9)
attention_mask                       (2, 9)
hidden (hình dạng lý thuyết)    (2, 9, 768)
logits (hình dạng lý thuyết)        (2, 28)
dtype: object

Không chạy encoder trong cell này; hidden/logit ở đây là hình dạng, không phải đầu ra đo được.


## 4. Không xóa dữ liệu khi biểu diễn không có đặc trưng TF-IDF
Vector rỗng vẫn đi qua classifier và thường chịu tác động chủ yếu của intercept. Bảng này đối chiếu số câu 46/9/12 trong Word.

In [5]:
from src.final_submission import restore_baselines
import joblib
artifacts=restore_baselines(artifacts_root())
model=joblib.load(artifacts/'data/processed/baseline/balanced/full/model.joblib')
rows=[]
for split,frame in frames.items():
    matrix=model.named_steps['tfidf'].transform(frame.text.tolist())
    rows.append({'split':split,'hình dạng ma trận':matrix.shape,'vector rỗng':int((matrix.getnnz(axis=1)==0).sum())})
display(pd.DataFrame(rows))
assert [r['vector rỗng'] for r in rows]==[46,9,12]
print('PASS: 58.338 đặc trưng và số vector rỗng khớp báo cáo.')

,split,hình dạng ma trận,vector rỗng
0,train,"(43410, 58338)",46
1,validation,"(5426, 58338)",9
2,test,"(5427, 58338)",12


PASS: 58.338 đặc trưng và số vector rỗng khớp báo cáo.
